##Synthetic Dataset## = **used car market dataset created**



## Used Car Financial Assets Dataset (~800k+)


## Import Libraries

In [ ]:
# Installing the libraries with the specified version.
!pip install numpy>=2.3 pandas==2.2.2 matplotlib==3.8.4 seaborn==0.13.2 -q --user

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import plotly.express as px

## Executive Plotly Theme

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"

BLUE_PRIMARY = "#174EA6"
BLUE_SECONDARY = "#4F8DF7"
BLUE_LIGHT = "#DCEBFF"
BLUE_DARK = "#0F2D5C"
GRID_COLOR = "#D9E2F3"
TEXT_COLOR = "#12355B"
PAPER_BG = "#F7FAFF"

EXEC_COLORS = [BLUE_PRIMARY, BLUE_SECONDARY, "#7FB3FF", "#A8C8FF", "#C7DBFF", "#E1ECFF"]

def apply_exec_style(fig, title_text=None):
    if title_text:
        fig.update_layout(title=title_text)
    fig.update_layout(
        paper_bgcolor=PAPER_BG,
        plot_bgcolor="white",
        font=dict(family="Arial", size=13, color=TEXT_COLOR),
        title=dict(font=dict(size=20, color=BLUE_DARK), x=0.02, xanchor="left"),
        margin=dict(l=40, r=30, t=70, b=40),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1,
            bgcolor="rgba(0,0,0,0)"
        ),
    )
    fig.update_xaxes(showgrid=False, linecolor=GRID_COLOR, tickfont=dict(color=TEXT_COLOR))
    fig.update_yaxes(showgrid=True, gridcolor=GRID_COLOR, zeroline=False, tickfont=dict(color=TEXT_COLOR))
    return fig

## Load and explore the data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
from google.colab import files

# Upload all 3 files at once
uploaded = files.upload()

# Read and combine uploaded files
df = pd.concat([
    pd.read_csv("used_car_financial_assets_800k_part1.csv"),
    pd.read_csv("used_car_financial_assets_800k_part2.csv"),
    pd.read_csv("used_car_financial_assets_800k_part3.csv")
], ignore_index=True)

print(df.shape)

Saving used_car_financial_assets_800k_part3.csv to used_car_financial_assets_800k_part3 (2).csv
Saving used_car_financial_assets_800k_part2.csv to used_car_financial_assets_800k_part2 (2).csv
Saving used_car_financial_assets_800k_part1.csv to used_car_financial_assets_800k_part1 (2).csv
(800000, 5)


## Daily append rows

Run this cell whenever you want to simulate a daily portfolio update.


In [ ]:
def generate_used_car_financial_assets_chunk(
    n_rows=100_000,
    start_asset_id=1,
    as_of_date=None,
    random_seed=42
):
    """
    Generate one chunk of synthetic used-car financial asset data

    Columns (5 total):
      1. asset_id
      2. vehicle_age_years
      3. mileage
      4. outstanding_loan_balance
      5. estimated_market_value
    """
    rng = np.random.default_rng(random_seed)

    if as_of_date is None:
        as_of_date = pd.Timestamp.today().normalize()
    else:
        as_of_date = pd.to_datetime(as_of_date)

    asset_id = np.arange(start_asset_id, start_asset_id + n_rows)

    # Used car age distribution: mostly 2 to 12 years, clipped to realistic range
    vehicle_age_years = np.clip(rng.normal(loc=6.5, scale=3.0, size=n_rows), 0, 18)

    # Mileage correlated with age
    base_miles_per_year = rng.normal(loc=12_500, scale=2_500, size=n_rows)
    mileage = np.clip(vehicle_age_years * base_miles_per_year + rng.normal(0, 8000, n_rows), 0, 280_000)

    # Market value decreases as age and mileage increase
    base_value = rng.normal(loc=24_000, scale=6_000, size=n_rows)
    depreciation_age = vehicle_age_years * rng.uniform(1200, 2200, size=n_rows)
    depreciation_mileage = (mileage / 1000) * rng.uniform(45, 90, size=n_rows)
    estimated_market_value = base_value - depreciation_age - depreciation_mileage + rng.normal(0, 1800, n_rows)
    estimated_market_value = np.clip(estimated_market_value, 1500, 60_000)

    # Loan balance should usually be related to market value but not identical
    ltv = np.clip(rng.normal(loc=0.82, scale=0.18, size=n_rows), 0.15, 1.40)
    outstanding_loan_balance = np.clip(estimated_market_value * ltv + rng.normal(0, 1500, n_rows), 0, 75_000)

    df_chunk = pd.DataFrame({
        'asset_id': asset_id.astype('int64'),
        'vehicle_age_years': np.round(vehicle_age_years, 1),
        'mileage': np.round(mileage).astype('int64'),
        'outstanding_loan_balance': np.round(outstanding_loan_balance, 2),
        'estimated_market_value': np.round(estimated_market_value, 2),
    })

    return df_chunk


def build_large_used_car_dataset(
    total_rows=800_000,
    chunk_size=100_000,
    output_path='used_car_financial_assets_800k.csv',
    start_asset_id=1,
    start_seed=42
):
    """
    Build a large dataset in chunks.
    """
    output_path = Path(output_path)

    if output_path.exists():
        output_path.unlink()  # remove old file so the rebuild is clean

    rows_written = 0
    chunk_num = 0

    while rows_written < total_rows:
        current_chunk_size = min(chunk_size, total_rows - rows_written)

        df_chunk = generate_used_car_financial_assets_chunk(
            n_rows=current_chunk_size,
            start_asset_id=start_asset_id + rows_written,
            random_seed=start_seed + chunk_num
        )

        df_chunk.to_csv(
            output_path,
            mode='a',
            index=False,
            header=(chunk_num == 0)
        )

        rows_written += current_chunk_size
        chunk_num += 1
        print(f"Wrote {rows_written:,} / {total_rows:,} rows")

    print(f"\nFinished: {output_path.resolve()}")
    return output_path


def append_daily_rows(
    existing_file='used_car_financial_assets_800k.csv',
    rows_to_add=10_000,
    random_seed=500
):
    """
    Append new rows daily to simulate portfolio growth or refreshed asset records
    """
    existing_file = Path(existing_file)

    if existing_file.exists():
        existing_df = pd.read_csv(existing_file, usecols=['asset_id'])
        next_asset_id = int(existing_df['asset_id'].max()) + 1
    else:
        next_asset_id = 1

    new_chunk = generate_used_car_financial_assets_chunk(
        n_rows=rows_to_add,
        start_asset_id=next_asset_id,
        random_seed=random_seed
    )

    new_chunk.to_csv(
        existing_file,
        mode='a',
        index=False,
        header=not existing_file.exists()
    )

    print(f"Appended {rows_to_add:,} rows starting at asset_id {next_asset_id:,}")
    return new_chunk.head()

In [ ]:

#  Dialy append 4,000 new rows
append_daily_rows(
    existing_file='used_car_financial_assets_800k.csv',
    rows_to_add=4_000,
    random_seed=999
)


Appended 4,000 rows starting at asset_id 800,001


,asset_id,vehicle_age_years,mileage,outstanding_loan_balance,estimated_market_value
0,800001,8.8,71031,4426.78,7298.96
1,800002,5.0,97815,3487.38,1500.00
2,800003,0.3,0,19864.15,23572.12
3,800004,6.5,90244,1338.58,10365.43
4,800005,8.1,152960,7471.54,8530.46


### Data Overview

In [ ]:
# Copy the data to another variable to avoid any changes to original data
data = df.copy()

In [ ]:
# Check the first 5 rows of the data
df.head()

,asset_id,vehicle_age_years,mileage,outstanding_loan_balance,estimated_market_value
0,1,7.4,84430,10331.32,15014.51
1,2,3.4,61337,20243.86,16364.52
2,3,8.8,144976,0.00,1500.00
3,4,9.3,88137,5041.10,5889.00
4,5,0.6,8633,15272.67,16135.18


In [ ]:
# Check the last 5 rows of the data
df.tail()

,asset_id,vehicle_age_years,mileage,outstanding_loan_balance,estimated_market_value
799995,799996,7.1,70416,5030.87,6654.82
799996,799997,3.2,63769,13656.79,14770.43
799997,799998,8.7,118242,3069.51,4966.49
799998,799999,6.0,118973,503.01,1500.00
799999,800000,5.4,65191,2924.58,1986.92


In [ ]:
# Check the shape and print the results
print(f"There are {len(df.axes[0])} rows and {len(df.axes[1])} columns.")

There are 800000 rows and 5 columns.


In [ ]:
# Check column types and number of values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800000 entries, 0 to 799999
Data columns (total 5 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   asset_id                  800000 non-null  int64  
 1   vehicle_age_years         800000 non-null  float64
 2   mileage                   800000 non-null  int64  
 3   outstanding_loan_balance  800000 non-null  float64
 4   estimated_market_value    800000 non-null  float64
dtypes: float64(3), int64(2)
memory usage: 30.5 MB


In [ ]:
# Get the statistical summary of the data
df.describe()

,asset_id,vehicle_age_years,mileage,outstanding_loan_balance,estimated_market_value
count,800000.000000,800000.000000,800000.000000,800000.000000,800000.000000
mean,400000.500000,6.512409,81455.020341,7626.431464,9234.707191
std,230940.252015,2.959793,41680.217226,6713.361127,7629.800653
min,1.000000,0.000000,0.000000,0.000000,1500.000000
25%,200000.750000,4.500000,51901.000000,2108.297500,1500.000000
50%,400000.500000,6.500000,78554.500000,5986.335000,7679.905000
75%,600000.250000,8.500000,107759.000000,11674.842500,14408.995000
max,800000.000000,18.000000,280000.000000,54875.330000,48178.080000


## Exploratory Data Analysis (EDA)

## Visualizations

In [51]:
# Executive-style box plot for vehicle_age_years
fig = px.box(
    df,
    x="vehicle_age_years",
    points=False,
    color_discrete_sequence=[BLUE_PRIMARY],
    title="Vehicle Age Distribution",
    labels={"vehicle_age_years": "Vehicle Age (Years)"}
)

fig.update_traces(
    fillcolor=BLUE_LIGHT,
    line=dict(color=BLUE_PRIMARY, width=2),
    marker=dict(color=BLUE_PRIMARY)
)

fig = apply_exec_style(fig)
fig.show()

Output hidden; open in https://colab.research.google.com to view.

* Outliers beyond 14.5 years to 18 years

In [ ]:
# Executive-style histogram for vehicle_age_years
fig = px.histogram(
    df,
    x="vehicle_age_years",
    nbins=20,
    color_discrete_sequence=[BLUE_PRIMARY],
    title="Vehicle Age Distribution",
    labels={"vehicle_age_years": "Vehicle Age (Years)"}
)

fig.update_traces(
    marker_line_color=BLUE_DARK,
    marker_line_width=1.2,
    opacity=0.9
)

fig.update_layout(
    xaxis_title="Vehicle Age (Years)",
    yaxis_title="Count of Vehicles"
)

fig = apply_exec_style(fig)
fig.show()

Output hidden; open in https://colab.research.google.com to view.

* The vehicle age is normally distributed

### Portfolio Monitoring

In [ ]:
# Total exposure
total_exposure = df["outstanding_loan_balance"].sum()
print(f"Total Exposure: ${total_exposure:,.2f}")

# Aggregate exposure by vehicle age
exposure_by_age = (
    df.groupby("vehicle_age_years")["outstanding_loan_balance"]
    .sum()
    .reset_index()
    .sort_values("vehicle_age_years")
)

# Executive-style exposure chart
fig = px.bar(
    exposure_by_age,
    x="vehicle_age_years",
    y="outstanding_loan_balance",
    title="Total Exposure by Vehicle Age",
    labels={
        "vehicle_age_years": "Vehicle Age (Years)",
        "outstanding_loan_balance": "Total Exposure ($)"
    },
    color_discrete_sequence=[BLUE_PRIMARY]
)

fig.update_traces(
    marker_line_color=BLUE_DARK,
    marker_line_width=1.2,
    hovertemplate="Vehicle Age: %{x}<br>Total Exposure: $%{y:,.0f}<extra></extra>"
)

fig = apply_exec_style(fig)
fig.show()

Total Exposure: $6,101,145,171.20


* The highest point for total expouse by vehicle age peaks at ~5.

### Depreciation Analysis

In [ ]:
# Create age bands
age_bins = [0, 3, 5, 7, 10, 15, 20]
age_labels = ["0-3", "4-5", "6-7", "8-10", "11-15", "16-20"]

df["age_band"] = pd.cut(
    df["vehicle_age_years"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True,
    right=True
)

# Aggregate average market value by age band
depreciation_by_age_band = (
    df.groupby("age_band", observed=False)["estimated_market_value"]
    .mean()
    .reset_index()
)

# Executive-style depreciation chart
fig = px.bar(
    depreciation_by_age_band,
    x="age_band",
    y="estimated_market_value",
    title="Average Estimated Market Value by Vehicle Age Band",
    labels={
        "age_band": "Vehicle Age Band (Years)",
        "estimated_market_value": "Average Market Value ($)"
    },
    text_auto=".2s",
    color_discrete_sequence=[BLUE_SECONDARY]
)

fig.update_traces(
    marker_line_color=BLUE_DARK,
    marker_line_width=1.2,
    textfont=dict(color=BLUE_DARK)
)

fig.update_layout(
    xaxis_title="Vehicle Age Band (Years)",
    yaxis_title="Average Market Value ($)"
)

fig = apply_exec_style(fig)
fig.show()

* The highest market value by vehicle age is at 20k where the age is three or less.

In [ ]:
print('''Project Folder/
│
├── app.py
├── used_car_financial_assets_800k.csv
└── pages/
    └── 1_Drill_Down.py''')

Project Folder/
│
├── app.py
├── used_car_financial_assets_800k.csv
└── pages/
    └── 1_Drill_Down.py


### Overview Page

In [ ]:
!pip install streamlit
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Executive blue palette
BLUE_PRIMARY = "#174EA6"
BLUE_SECONDARY = "#4F8DF7"
BLUE_LIGHT = "#DCEBFF"
BLUE_DARK = "#0F2D5C"
GRID_COLOR = "#D9E2F3"
TEXT_COLOR = "#12355B"
PAPER_BG = "#F7FAFF"
EXEC_COLORS = [BLUE_PRIMARY, BLUE_SECONDARY, "#7FB3FF", "#A8C8FF", "#C7DBFF", "#E1ECFF"]

def apply_exec_style(fig):
    fig.update_layout(
        paper_bgcolor=PAPER_BG,
        plot_bgcolor="white",
        font=dict(family="Arial", size=13, color=TEXT_COLOR),
        title=dict(font=dict(size=20, color=BLUE_DARK), x=0.02, xanchor="left"),
        margin=dict(l=40, r=30, t=70, b=40),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    fig.update_xaxes(showgrid=False, linecolor=GRID_COLOR)
    fig.update_yaxes(showgrid=True, gridcolor=GRID_COLOR, zeroline=False)
    return fig

st.set_page_config(
    page_title="Used Car Market Dashboard",
    page_icon="🚗",
    layout="wide"
)

st.markdown(
    f"""
    <style>
    .stApp {{
        background-color: {PAPER_BG};
    }}
    div[data-testid="stMetric"] {{
        background-color: white;
        border: 1px solid {GRID_COLOR};
        padding: 14px;
        border-radius: 12px;
        box-shadow: 0 2px 6px rgba(15,45,92,.08);
    }}
    </style>
    """,
    unsafe_allow_html=True
)

st.title("🚗 Used Car Market Dashboard")
st.subheader("Executive Overview")

@st.cache_data
def load_data():
    return pd.read_csv("used_car_financial_assets_800k.csv")

data.to_csv("used_car_financial_assets_800k.csv", index=False)
df = load_data()

df["negative_equity"] = df["outstanding_loan_balance"] > df["estimated_market_value"]
df["equity_gap"] = df["outstanding_loan_balance"] - df["estimated_market_value"]

age_bins = [0, 3, 5, 7, 10, 15, 20]
age_labels = ["0-3", "4-5", "6-7", "8-10", "11-15", "16-20"]

df["age_band"] = pd.cut(df["vehicle_age_years"], bins=age_bins, labels=age_labels, include_lowest=True)

total_assets = len(df)
total_exposure = df["outstanding_loan_balance"].sum()
avg_market_value = df["estimated_market_value"].mean()
avg_ltv = (df["outstanding_loan_balance"] / df["estimated_market_value"]).replace([float("inf")], pd.NA).dropna().mean()
negative_equity_rate = df["negative_equity"].mean() * 100

col1, col2, col3, col4, col5 = st.columns(5)
col1.metric("Total Assets", f"{total_assets:,}")
col2.metric("Total Exposure", f"${total_exposure:,.0f}")
col3.metric("Avg Market Value", f"${avg_market_value:,.0f}")
col4.metric("Avg LTV", f"{avg_ltv:.2f}x")
col5.metric("Negative Equity Rate", f"{negative_equity_rate:.1f}%")

st.markdown("---")

exposure_by_age = df.groupby("age_band", observed=False)["outstanding_loan_balance"].sum().reset_index()
fig_exposure = px.bar(
    exposure_by_age, x="age_band", y="outstanding_loan_balance",
    title="Total Exposure by Vehicle Age Band",
    labels={"age_band":"Vehicle Age Band", "outstanding_loan_balance":"Total Exposure ($)"},
    color_discrete_sequence=[BLUE_PRIMARY]
)
fig_exposure.update_traces(marker_line_color=BLUE_DARK, marker_line_width=1.2)
fig_exposure = apply_exec_style(fig_exposure)

value_by_age = df.groupby("age_band", observed=False)["estimated_market_value"].mean().reset_index()
fig_value = px.line(
    value_by_age, x="age_band", y="estimated_market_value", markers=True,
    title="Average Market Value by Vehicle Age Band",
    labels={"age_band":"Vehicle Age Band", "estimated_market_value":"Average Market Value ($)"},
    color_discrete_sequence=[BLUE_SECONDARY]
)
fig_value.update_traces(line=dict(width=4), marker=dict(size=9, color=BLUE_PRIMARY))
fig_value = apply_exec_style(fig_value)

comparison_df = df.groupby("age_band", observed=False)[["estimated_market_value", "outstanding_loan_balance"]].mean().reset_index()
comparison_long = comparison_df.melt(
    id_vars="age_band",
    value_vars=["estimated_market_value", "outstanding_loan_balance"],
    var_name="metric",
    value_name="amount"
)
fig_compare = px.line(
    comparison_long, x="age_band", y="amount", color="metric", markers=True,
    title="Average Market Value vs Loan Balance by Age Band",
    labels={"age_band":"Vehicle Age Band", "amount":"Average Amount ($)", "metric":"Metric"},
    color_discrete_sequence=[BLUE_PRIMARY, BLUE_SECONDARY]
)
fig_compare.update_traces(line=dict(width=4), marker=dict(size=8))
fig_compare = apply_exec_style(fig_compare)

negative_equity_df = df.groupby("age_band", observed=False)["negative_equity"].mean().reset_index()
negative_equity_df["negative_equity_pct"] = negative_equity_df["negative_equity"] * 100
fig_negative = px.bar(
    negative_equity_df, x="age_band", y="negative_equity_pct",
    title="Negative Equity Rate by Age Band",
    labels={"age_band":"Vehicle Age Band", "negative_equity_pct":"Negative Equity Rate (%)"},
    color_discrete_sequence=[BLUE_SECONDARY]
)
fig_negative.update_traces(marker_line_color=BLUE_DARK, marker_line_width=1.2)
fig_negative = apply_exec_style(fig_negative)

left_col, right_col = st.columns(2)
with left_col:
    st.plotly_chart(fig_exposure, use_container_width=True)
    st.plotly_chart(fig_compare, use_container_width=True)
with right_col:
    st.plotly_chart(fig_value, use_container_width=True)
    st.plotly_chart(fig_negative, use_container_width=True)

st.markdown("---")
st.write(
    "This executive overview summarizes portfolio size, exposure, depreciation patterns, "
    "and negative equity risk across the used car asset portfolio."
)

2026-04-05 23:01:05.077 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 23:01:05.079 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 23:01:05.081 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 23:01:05.083 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 23:01:05.085 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 23:01:05.086 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 23:01:05.088 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 23:01:05.090 No runtime found, using MemoryCacheStorageManager
2026-04-05 23:01:08.022 Thread 'MainThread':

### Drill-Down Page

In [ ]:
import streamlit as st
import pandas as pd
import plotly.express as px

BLUE_PRIMARY = "#174EA6"
BLUE_SECONDARY = "#4F8DF7"
BLUE_LIGHT = "#DCEBFF"
BLUE_DARK = "#0F2D5C"
GRID_COLOR = "#D9E2F3"
TEXT_COLOR = "#12355B"
PAPER_BG = "#F7FAFF"
EXEC_COLORS = [BLUE_PRIMARY, BLUE_SECONDARY, "#7FB3FF", "#A8C8FF", "#C7DBFF", "#E1ECFF"]

def apply_exec_style(fig):
    fig.update_layout(
        paper_bgcolor=PAPER_BG,
        plot_bgcolor="white",
        font=dict(family="Arial", size=13, color=TEXT_COLOR),
        title=dict(font=dict(size=20, color=BLUE_DARK), x=0.02, xanchor="left"),
        margin=dict(l=40, r=30, t=70, b=40),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    fig.update_xaxes(showgrid=False, linecolor=GRID_COLOR)
    fig.update_yaxes(showgrid=True, gridcolor=GRID_COLOR, zeroline=False)
    return fig

st.set_page_config(page_title="Used Car Market Drill-Down", page_icon="🔍", layout="wide")

st.markdown(
    f"""
    <style>
    .stApp {{
        background-color: {PAPER_BG};
    }}
    div[data-testid="stMetric"] {{
        background-color: white;
        border: 1px solid {GRID_COLOR};
        padding: 14px;
        border-radius: 12px;
        box-shadow: 0 2px 6px rgba(15,45,92,.08);
    }}
    </style>
    """,
    unsafe_allow_html=True
)

st.title("🔍 Used Car Market Dashboard")
st.subheader("Executive Drill-Down")

@st.cache_data
def load_data():
    return pd.read_csv("used_car_financial_assets_800k.csv")

df = load_data()
df["negative_equity"] = df["outstanding_loan_balance"] > df["estimated_market_value"]
df["equity_gap"] = df["outstanding_loan_balance"] - df["estimated_market_value"]

age_bins = [0, 3, 5, 7, 10, 15, 20]
age_labels = ["0-3", "4-5", "6-7", "8-10", "11-15", "16-20"]
df["age_band"] = pd.cut(df["vehicle_age_years"], bins=age_bins, labels=age_labels, include_lowest=True)

st.sidebar.header("Filter Options")
selected_age_bands = st.sidebar.multiselect(
    "Select Age Band(s)",
    options=df["age_band"].dropna().unique(),
    default=df["age_band"].dropna().unique()
)

mileage_range = st.sidebar.slider(
    "Select Mileage Range",
    min_value=int(df["mileage"].min()),
    max_value=int(df["mileage"].max()),
    value=(int(df["mileage"].min()), int(df["mileage"].max()))
)

market_value_range = st.sidebar.slider(
    "Select Market Value Range",
    min_value=float(df["estimated_market_value"].min()),
    max_value=float(df["estimated_market_value"].max()),
    value=(float(df["estimated_market_value"].min()), float(df["estimated_market_value"].max()))
)

negative_equity_filter = st.sidebar.selectbox("Negative Equity", options=["All", "Yes", "No"])

filtered_df = df[df["age_band"].isin(selected_age_bands)]
filtered_df = filtered_df[(filtered_df["mileage"] >= mileage_range[0]) & (filtered_df["mileage"] <= mileage_range[1])]
filtered_df = filtered_df[
    (filtered_df["estimated_market_value"] >= market_value_range[0]) &
    (filtered_df["estimated_market_value"] <= market_value_range[1])
]

if negative_equity_filter == "Yes":
    filtered_df = filtered_df[filtered_df["negative_equity"]]
elif negative_equity_filter == "No":
    filtered_df = filtered_df[~filtered_df["negative_equity"]]

total_assets = len(filtered_df)
total_exposure = filtered_df["outstanding_loan_balance"].sum()
avg_market_value = filtered_df["estimated_market_value"].mean() if len(filtered_df) > 0 else 0
negative_equity_rate = filtered_df["negative_equity"].mean() * 100 if len(filtered_df) > 0 else 0

col1, col2, col3, col4 = st.columns(4)
col1.metric("Filtered Assets", f"{total_assets:,}")
col2.metric("Filtered Exposure", f"${total_exposure:,.0f}")
col3.metric("Avg Market Value", f"${avg_market_value:,.0f}")
col4.metric("Negative Equity Rate", f"{negative_equity_rate:.1f}%")

st.markdown("---")

sample_df = filtered_df.sample(min(len(filtered_df), 10000), random_state=42) if len(filtered_df) > 0 else filtered_df

scatter_fig = px.scatter(
    sample_df,
    x="estimated_market_value",
    y="outstanding_loan_balance",
    color="age_band",
    color_discrete_sequence=EXEC_COLORS,
    hover_data=["asset_id", "vehicle_age_years", "mileage"],
    title="Loan Balance vs Market Value",
    labels={
        "estimated_market_value": "Estimated Market Value ($)",
        "outstanding_loan_balance": "Outstanding Loan Balance ($)"
    },
    opacity=0.65
)
scatter_fig.update_traces(marker=dict(size=8, line=dict(width=0.5, color=BLUE_DARK)))
scatter_fig = apply_exec_style(scatter_fig)

hist_fig = px.histogram(
    filtered_df,
    x="vehicle_age_years",
    nbins=20,
    title="Distribution of Vehicle Age",
    labels={"vehicle_age_years": "Vehicle Age (Years)"},
    color_discrete_sequence=[BLUE_PRIMARY]
)
hist_fig.update_traces(marker_line_color=BLUE_DARK, marker_line_width=1.2)
hist_fig = apply_exec_style(hist_fig)

equity_gap_fig = px.box(
    filtered_df,
    x="age_band",
    y="equity_gap",
    title="Equity Gap by Age Band",
    labels={"age_band": "Vehicle Age Band", "equity_gap": "Loan Balance - Market Value ($)"},
    color_discrete_sequence=[BLUE_SECONDARY]
)
equity_gap_fig.update_traces(fillcolor=BLUE_LIGHT, line=dict(color=BLUE_PRIMARY, width=2))
equity_gap_fig = apply_exec_style(equity_gap_fig)

left_col, right_col = st.columns(2)
with left_col:
    st.plotly_chart(scatter_fig, use_container_width=True)
    st.plotly_chart(hist_fig, use_container_width=True)
with right_col:
    st.plotly_chart(equity_gap_fig, use_container_width=True)

st.markdown("---")
st.subheader("Filtered Asset Detail")
display_cols = [
    "asset_id", "vehicle_age_years", "mileage", "outstanding_loan_balance",
    "estimated_market_value", "negative_equity", "equity_gap"
]
st.dataframe(filtered_df[display_cols].head(500), use_container_width=True)

csv = filtered_df[display_cols].to_csv(index=False).encode("utf-8")
st.download_button(
    label="Download Filtered Data as CSV",
    data=csv,
    file_name="filtered_used_car_assets.csv",
    mime="text/csv"
)

2026-04-05 23:01:23.081 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 23:01:23.093 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 23:01:23.101 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 23:01:23.106 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 23:01:23.116 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 23:01:23.123 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 23:01:23.128 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 23:01:23.139 No runtime found, using MemoryCacheStorageManager
2026-04-05 23:01:23.356 Thread 'MainThread':

False